# 01 · Exploración de los archivos del ICT

Objetivo del día 1: saber qué años, países, vías y problemas trae cada archivo.

In [1]:
from pathlib import Path

RAW = Path("../data/raw")
sorted(p.name for p in RAW.glob("*.pdf"))

['2025-12_Llegadas_internacionales.pdf', 'Anuario_de_Turismo_2024.pdf']

In [2]:
# Ver un cuadro tal como sale del PDF
import sys

sys.path.append("..")
from src.limpieza import extraer_cuadro_pdf, leer_cuadro, leer_serie, validar  # noqa: E402

archivo = RAW / "2025-12_Llegadas_internacionales.pdf"
grilla = extraer_cuadro_pdf(archivo, pagina=8, cuadro=7)  # LIR
grilla.head(20)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Setiembre,Octubre,Noviembre,Diciembre,Ene-Dic
1,TOTAL,101846,96210,117044,81894,61961,77274,79891,49477,22176,30779,71524,114686,904762
2,AMÉRICA DEL NORTE,97866,92783,113365,77983,59282,75472,76998,47252,21231,29559,68617,109945,870353
3,Canadá,27645,25137,24683,15294,6173,4966,7040,5851,2517,5634,19365,29313,173618
4,Estados Unidos,70084,67545,88551,62531,53001,70360,69793,41319,18628,23862,49123,80476,695273
5,México,137,101,131,158,108,146,165,82,86,63,129,156,1462
6,AMÉRICA CENTRAL,57,53,43,56,56,60,67,44,87,24,60,103,710
7,Belice,None,None,None,None,1,1,1,None,1,1,1,1,7
8,El Salvador,18,23,16,7,20,18,9,15,5,8,12,20,171
9,Guatemala,18,8,5,5,18,8,19,9,9,6,11,22,138


In [3]:
# Limpiar y validar contra la fila TOTAL
datos, totales = leer_cuadro(grilla, anio=2025, via="lir")
validar(datos, totales)
datos.groupby("region")["llegadas"].sum().sort_values(ascending=False)

region
América del Norte     870353
Europa                 22681
Asia y el Pacífico      7393
América del Sur         2072
Caribe                   749
América Central          710
África                   566
Oriente Medio            238
Name: llegadas, dtype: int64

In [4]:
# Serie 2019-2025 de LIR
serie = leer_serie(extraer_cuadro_pdf(archivo, pagina=13, cuadro=14), via="lir")
serie.groupby(serie["fecha"].dt.year)["llegadas"].sum()

fecha
2019    599433
2020    228728
2021    388054
2022    661579
2023    769723
2024    881289
2025    904762
Name: llegadas, dtype: int64

## Notas

- Formato: PDF, un informe por año. Tablas con texto (no escaneadas).
- Detalle por país: solo 2025 en este informe. 72 países, 8 regiones.
- Vías: todas, aérea, SJO, LIR, terrestre. No hay marítima.
- Nombres en MAYÚSCULAS en cuadros 1-6, en minúsculas con tildes en 7-10.
- Celdas vacías = sin dato (ej. Guyana Francesa).
- Páginas 14-15 son gráficos.

## Hallazgos

Las cifras de abajo salen de `data/processed/turismo.duckdb` con las mismas funciones que usa la app (`src/consultas.py` e `src/indicadores.py`). Son las del README.

In [5]:
import sys

sys.path.append("..")
import duckdb  # noqa: E402

from src import consultas as q  # noqa: E402
from src import indicadores as ind  # noqa: E402
from src.estilo import MESES_CORTOS, fmt  # noqa: E402

con = duckdb.connect("../data/processed/turismo.duckdb", read_only=True)
SIN_DETALLE = dict(regiones=[], paises=[])

### 1. Por aire el país ya superó 2019; por tierra no

In [6]:
def acumulado_2025(via):
    serie = q.serie_mensual(con, via=via, desde=2019, hasta=2025, **SIN_DETALLE)
    return ind.acumulado_vs_2019(serie, 2025), serie


for nombre, via in [
    ("Todas las vías", "todas"),
    ("Aérea", "aerea"),
    ("Terrestre", "terrestre"),
    ("LIR", "lir"),
]:
    (total, variacion, meses), serie = acumulado_2025(via)
    print(f"{nombre}: {fmt(total)} llegadas en 2025 ({meses} meses), {fmt(variacion, 'pct')} vs 2019")

diciembre = serie[serie["fecha"] == "2025-12-01"].iloc[0]  # `serie` quedó con LIR
print(
    f"LIR en diciembre de 2025: {fmt(diciembre['llegadas'])} ({fmt(diciembre['vs_2019'], 'pct')} vs diciembre de 2019)"
)

Todas las vías: 2.943.991 llegadas en 2025 (12 meses), -6,2% vs 2019
Aérea: 2.689.278 llegadas en 2025 (12 meses), +11,2% vs 2019
Terrestre: 246.184 llegadas en 2025 (12 meses), -65,3% vs 2019
LIR: 904.762 llegadas en 2025 (12 meses), +50,9% vs 2019
LIR en diciembre de 2025: 114.686 (+63,5% vs diciembre de 2019)


En 2025 llegaron 2.943.991 turistas por todas las vías, 6,2% menos que en 2019. La vía aérea superó 2019 y la terrestre quedó muy por debajo, y eso explica el total. LIR recibió 904.762 turistas (+50,9% contra 2019).

### 2. Estados Unidos y Canadá sostienen a LIR; Europa cae

In [7]:
filtros = dict(via="lir", desde=2025, hasta=2025, **SIN_DETALLE)

serie = q.serie_mensual(con, via="lir", desde=2024, hasta=2025, **SIN_DETALLE)
por_anio = serie.groupby(serie["fecha"].dt.year)["llegadas"].sum()
print(f"LIR 2025 contra 2024: {fmt(por_anio[2025] / por_anio[2024] - 1, 'pct')}")

ranking = q.ranking_paises(con, **filtros).set_index("pais")
for pais in ["Estados Unidos", "Canadá", "Reino Unido"]:
    fila = ranking.loc[pais]
    print(f"{pais}: {fmt(fila['llegadas'])} llegadas, {fmt(fila['crecimiento'], 'pct')}")

crece = q.mas_crece(ranking.reset_index())
print(f"Mayor crecimiento entre los 10 principales: {crece['pais']} ({fmt(crece['crecimiento'], 'pct')})")

idioma = q.idiomas_principales(con, **filtros).iloc[0]
print(f"Idioma principal: {idioma['idioma']} ({fmt(idioma['participacion'], 'part')} de las llegadas)")

LIR 2025 contra 2024: +2,7%
Estados Unidos: 695.273 llegadas, +2,3%
Canadá: 173.618 llegadas, +6,2%
Reino Unido: 7.930 llegadas, -22,8%
Mayor crecimiento entre los 10 principales: Canadá (+6,2%)
Idioma principal: inglés (97,5% de las llegadas)


LIR creció 2,7% contra 2024. Estados Unidos lidera con 695.273 llegadas (+2,3%) y Canadá es el que más crece entre los 10 principales (+6,2%). Reino Unido bajó 22,8%, y el inglés es el idioma de 97,5% de las llegadas.

### 3. La demanda pica en marzo y toca fondo en setiembre

In [8]:
serie = q.serie_mensual(con, via="lir", desde=2019, hasta=2025, **SIN_DETALLE)
concentracion = ind.concentracion_temporada(serie)
perfil = ind.perfil_mensual(serie)
pico, valle = perfil.idxmax(), perfil.idxmin()

print(
    f"Diciembre a abril concentra {fmt(concentracion, 'part')} de las llegadas (años completos, sin 2020 ni 2021)"
)
print(f"Mes pico: {MESES_CORTOS[pico - 1]}. Mes más bajo: {MESES_CORTOS[valle - 1]}.")
print(f"El pico tiene {fmt(perfil[pico] / perfil[valle], 'dec')} veces las llegadas del mes más bajo")

Diciembre a abril concentra 55,0% de las llegadas (años completos, sin 2020 ni 2021)
Mes pico: Mar. Mes más bajo: Set.
El pico tiene 4,2 veces las llegadas del mes más bajo


Con años completos y sin 2020 y 2021, diciembre a abril concentra 55,0% de las llegadas. Marzo tiene 4,2 veces las llegadas de setiembre. Semana Santa cambia de mes (2024 en marzo, 2025 en abril) y mueve las comparaciones de marzo y abril.